# Grupo 6 – Base Adult – KNN, SVM e MLP
**Inteligência de Negócios – BSI – 2026-2**

Notebook do grupo. Seções da **Pessoa 1**: análise exploratória (3.1), pré-processamento e validação (3.2), KNN e SVM.
A **Pessoa 2** adiciona o MLP usando exatamente o mesmo `prep`, `skf` e `avaliar` (seção reservada mais abaixo).

> Para os tempos serem comparáveis entre modelos, **o notebook inteiro deve ser executado no mesmo ambiente (Google Colab)**.

## A1. Ambiente

In [ ]:
# No Colab, só falta o ucimlrepo (pandas, sklearn, seaborn etc. já vêm instalados)
%pip install -q ucimlrepo

In [ ]:
import os, json, time, platform
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import sklearn

os.makedirs('figuras', exist_ok=True)
os.makedirs('resultados', exist_ok=True)
SEED = 42
print('Python', platform.python_version(), '| sklearn', sklearn.__version__,
      '| pandas', pd.__version__, '| CPUs', os.cpu_count())

## A2. Análise exploratória (requisito 3.1)
### Carregamento
A base é baixada do UCI (id=2: 32.561 registros de treino + 16.281 de teste, unidos). Uma cópia local
(`adult_raw.csv`) é salva para que o notebook não dependa do servidor do UCI.

In [ ]:
if os.path.exists('adult_raw.csv'):
    df = pd.read_csv('adult_raw.csv')
else:
    from ucimlrepo import fetch_ucirepo
    adult = fetch_ucirepo(id=2)
    df = pd.concat([adult.data.features, adult.data.targets], axis=1)
    df.to_csv('adult_raw.csv', index=False)

print('Ausentes já como NaN no arquivo original:')
print(df.isna().sum()[lambda s: s > 0])

### Limpeza de marcações
- Espaços extras nos textos são removidos.
- O rótulo vem em duas grafias (`<=50K` no treino e `<=50K.` no teste) – unificamos.
- A base marca ausentes **de duas formas**: `NaN` e `'?'`. Convertemos `'?'` para `NaN`.

In [ ]:
for c in df.select_dtypes(exclude='number').columns:
    df[c] = df[c].str.strip()
df['income'] = df['income'].str.rstrip('.')   # '<=50K.' e '<=50K' viram a mesma classe
df = df.replace('?', np.nan)                 # a base usa '?' para ausente

### 1) Registros, atributos e tipos

In [ ]:
print('Registros x colunas:', df.shape)
print(df.dtypes)

### 2) Variável-alvo
`income`: `<=50K` ou `>50K` (renda anual). Classificação binária; a classe positiva (1) será `>50K`.

### 3) Ausentes, duplicados e inconsistências

In [ ]:
aus = df.isna().sum()
print(pd.DataFrame({'ausentes': aus, '%': (100 * aus / len(df)).round(2)})[aus > 0])
print('\nRegistros com pelo menos um ausente:', df.isna().any(axis=1).sum())
print('Duplicados exatos:', df.duplicated().sum())

In [ ]:
df.describe().T

In [ ]:
# capital-gain = 99999 é o teto de registro do censo (valor truncado), não erro de digitação.
print('Registros com capital-gain = 99999:', (df['capital-gain'] == 99999).sum())
print('% de zeros em capital-gain:', round(100 * (df['capital-gain'] == 0).mean(), 1))
print('% de zeros em capital-loss:', round(100 * (df['capital-loss'] == 0).mean(), 1))
# education e education-num: relação 1 para 1?
print('Valores de education-num por categoria de education (máx):',
      df.groupby('education')['education-num'].nunique().max())

### 4) Distribuição da variável-alvo

In [ ]:
print(df['income'].value_counts())
print(df['income'].value_counts(normalize=True).round(4))
ax = df['income'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'], rot=0)
ax.set_title('Distribuição de income'); ax.set_ylabel('Registros')
plt.tight_layout(); plt.savefig('figuras/dist_income.png', dpi=300); plt.show()

### Figuras complementares

In [ ]:
# Histogramas de age, hours-per-week e education-num
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, c in zip(axs, ['age', 'hours-per-week', 'education-num']):
    sns.histplot(data=df, x=c, hue='income', bins=30, ax=ax, multiple='stack')
    ax.set_title(c)
plt.tight_layout(); plt.savefig('figuras/histogramas.png', dpi=300); plt.show()

In [ ]:
# Proporção de >50K em algumas categóricas
cats = ['sex', 'relationship', 'marital-status', 'workclass']
fig, axs = plt.subplots(2, 2, figsize=(14, 9))
for ax, c in zip(axs.ravel(), cats):
    prop = pd.crosstab(df[c], df['income'], normalize='index').sort_values('>50K')
    prop.plot(kind='barh', stacked=True, ax=ax, color=['#4C72B0', '#DD8452'], legend=False)
    ax.set_title(f'income por {c}'); ax.set_xlabel('proporção'); ax.set_ylabel('')
axs[0, 0].legend(title='income', loc='lower right')
plt.tight_layout(); plt.savefig('figuras/categoricas_por_income.png', dpi=300); plt.show()

In [ ]:
# Boxplot das numéricas por income
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, c in zip(axs, ['age', 'hours-per-week', 'education-num']):
    sns.boxplot(data=df, x='income', y=c, ax=ax)
plt.tight_layout(); plt.savefig('figuras/boxplots.png', dpi=300); plt.show()

In [ ]:
# Matriz de correlação das variáveis numéricas
num = df.select_dtypes(include='number')
plt.figure(figsize=(7, 5.5))
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlação (Pearson) das variáveis numéricas')
plt.tight_layout(); plt.savefig('figuras/correlacao.png', dpi=300); plt.show()

### Conclusão da EDA
- **48.842 registros e 15 colunas** (6 numéricas, 8 categóricas e o alvo).
- **Desbalanceamento:** ~76,1% `<=50K` e ~23,9% `>50K`. Um classificador que sempre prevê `<=50K` já teria
  ~76% de acurácia, por isso precisão, revocação e F1 da classe `>50K` são essenciais na interpretação.
- **Ausentes** apenas em `workclass` (2.799), `occupation` (2.809) e `native-country` (857).
- **52 registros duplicados exatos.**
- `capital-gain` e `capital-loss` são muito assimétricas (≈92% e ≈95% de zeros); 244 registros têm
  `capital-gain = 99999`, que é o teto do censo – mantidos, pois representam rendas reais muito altas.
- `education` e `education-num` codificam a mesma informação (relação 1 para 1).
- `fnlwgt` tem correlação praticamente nula com as demais numéricas – é um peso amostral do censo.

## A3. Pré-processamento e validação (requisito 3.2)

| Decisão | Justificativa |
|---|---|
| Remover duplicados **antes** de remover `fnlwgt` | Sem `fnlwgt`, pessoas diferentes ficariam parecendo iguais. Só os duplicados exatos saem. |
| Remover `fnlwgt` | Peso amostral do censo, não uma característica da pessoa. |
| Remover `education` | Redundante com `education-num`. |
| Imputar ausentes: moda (categóricas) e mediana (numéricas) | Mantém todos os registros. |
| One-hot nas categóricas | KNN, SVM e MLP não lidam com texto. |
| StandardScaler nas numéricas | KNN e SVM dependem de distância; o MLP converge melhor com dados padronizados. |
| Tudo dentro de um `Pipeline` | Imputador, scaler e encoder são ajustados **só no treino de cada fold** (sem vazamento de dados). |
| `StratifiedKFold(5, shuffle=True, random_state=42)` | **Mesma divisão para todos os modelos**, mantendo a proporção das classes em cada fold. |

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = df.drop_duplicates()                      # 1º: duplicados exatos
df = df.drop(columns=['fnlwgt', 'education'])  # 2º: colunas inapropriadas/redundantes

y = (df['income'] == '>50K').astype(int)       # 1 = classe positiva (>50K)
X = df.drop(columns='income').reset_index(drop=True)
y = y.reset_index(drop=True)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
])

# MESMA divisão para TODOS os modelos
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('Após limpeza:', X.shape, '| numéricas:', num_cols)
print('Categóricas:', cat_cols)
print('Proporção >50K:', round(y.mean(), 4))
print('Atributos após one-hot (ajustado na base toda só para contar):',
      prep.fit_transform(X).shape[1])

In [ ]:
# Registro dos folds (prova de que todos os modelos usam a mesma divisão)
fold_id = np.empty(len(X), dtype=int)
for i, (tr, te) in enumerate(skf.split(X, y)):
    fold_id[te] = i
    print(f'Fold {i}: treino={len(tr)}  teste={len(te)}  %>50K no teste={y.iloc[te].mean():.4f}')
pd.DataFrame({'fold': fold_id}).to_csv('resultados/folds.csv', index_label='linha')

## A4. Função de avaliação (compartilhada com a Pessoa 2)
Um único treino por fold; coleta acurácia, precisão, revocação, F1 (da classe positiva `>50K`) e a matriz
de confusão somada dos 5 folds. O desvio padrão é o populacional (`np.std`, ddof=0) entre os 5 folds.

`max_treino` (opcional, padrão `None` = usa tudo): **somente** como último recurso para o SVM RBF, reduz o
**treino** de cada fold a uma amostra estratificada; o **teste continua sendo o fold completo**, então os folds
continuam os mesmos dos demais modelos. Se for usado, tem que ser declarado no artigo.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

resultados = []   # lista de dicionários (vira tabela)
matrizes = {}     # nome -> matriz de confusão somada dos 5 folds

def avaliar(nome, clf, max_treino=None):
    pipe = Pipeline([('prep', prep), ('clf', clone(clf))])
    cm_total = np.zeros((2, 2), dtype=int)
    metr = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
    t0 = time.time()
    for tr, te in skf.split(X, y):
        if max_treino is not None and len(tr) > max_treino:
            tr, _ = train_test_split(tr, train_size=max_treino, stratify=y.iloc[tr],
                                     random_state=SEED)
        pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = pipe.predict(X.iloc[te])
        metr['acc'].append(accuracy_score(y.iloc[te], pred))
        metr['prec'].append(precision_score(y.iloc[te], pred, zero_division=0))
        metr['rec'].append(recall_score(y.iloc[te], pred))
        metr['f1'].append(f1_score(y.iloc[te], pred))
        cm_total += confusion_matrix(y.iloc[te], pred)
    linha = {'modelo': nome, 'tempo_s': round(time.time() - t0, 1),
             'treino_por_fold': max_treino or 'completo'}
    for k, v in metr.items():
        linha[k + '_media'] = np.mean(v)
        linha[k + '_dp'] = np.std(v)
    resultados.append(linha)
    matrizes[nome] = cm_total
    print(f"{nome}: acc={linha['acc_media']:.4f} prec={linha['prec_media']:.4f} "
          f"rec={linha['rec_media']:.4f} f1={linha['f1_media']:.4f} ({linha['tempo_s']}s)")

## A5. KNN
Varia o número de vizinhos `k` (5, 15, 31) com voto uniforme e compara com o voto ponderado pela distância
(k=15). k pequeno tende a overfitting (fronteira ruidosa); k grande suaviza demais.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in [5, 15, 31]:
    avaliar(f"KNN k={k} uniform", KNeighborsClassifier(n_neighbors=k, weights='uniform', n_jobs=-1))
avaliar("KNN k=15 distance", KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=-1))

**Interpretação (KNN).** O F1 da classe `>50K` sobe de k=5 para k=15 e praticamente estabiliza em k=31
(diferença de ~0,001, bem menor que o desvio padrão entre folds, ~0,006). Com k=5 a fronteira é ruidosa
(overfitting): precisão menor e mais falsos positivos. O voto ponderado pela distância (k=15) ficou **abaixo** do
voto uniforme com o mesmo k, porque dá peso excessivo aos vizinhos mais próximos e se comporta como um k menor.
**Melhor configuração: KNN k=31 uniform** (k=15 uniform é praticamente equivalente).

## A6. SVM
Kernel RBF com C=1 e C=10 (C maior = margem mais rígida, menos regularização) e um SVM linear como
comparação rápida. O SVM RBF é a etapa mais lenta do trabalho (vários minutos por fold).
`cache_size` maior só acelera o treino, não muda o resultado.

In [ ]:
from sklearn.svm import SVC, LinearSVC

avaliar("SVM RBF C=1",    SVC(kernel='rbf', C=1,  gamma='scale', cache_size=2000))
avaliar("SVM RBF C=10",   SVC(kernel='rbf', C=10, gamma='scale', cache_size=2000))
avaliar("SVM Linear C=1", LinearSVC(C=1, max_iter=5000))   # rápido, útil como comparação

**Interpretação (SVM).** O RBF com C=1 teve a maior acurácia e precisão; com C=10 (menos regularização) a
revocação e o F1 sobem um pouco, mas as diferenças estão dentro do desvio padrão entre folds, e o treino fica
mais lento. O SVM linear chega perto do RBF (F1 ~0,655 contra ~0,666) com um custo computacional centenas de
vezes menor, o que indica que, após o one-hot, as classes são quase linearmente separáveis.
**Melhor configuração pelo F1: SVM RBF C=10** (C=1 se o critério for acurácia).

Em todos os modelos a revocação da classe `>50K` fica em ~0,59–0,61: cerca de 40% das pessoas de renda alta são
classificadas como `<=50K`. Os erros são dominados por **falsos negativos**, efeito do desbalanceamento (76/24).

## Seção da Pessoa 2: MLP
*(Reservada. Adicionar aqui as células do MLP usando `avaliar(...)`, sem redefinir `prep`, `skf` ou `avaliar`.)*

## A7. Exportação dos resultados

In [ ]:
tab = pd.DataFrame(resultados)
tab.to_csv('resultados/resultados.csv', index=False)
with open('resultados/matrizes.json', 'w') as f:
    # formato: [[TN, FP], [FN, TP]]  (linhas = real, colunas = previsto; 0 = <=50K, 1 = >50K)
    json.dump({k: v.tolist() for k, v in matrizes.items()}, f, indent=1)

pd.set_option('display.width', 200)
print(tab[['modelo', 'acc_media', 'acc_dp', 'prec_media', 'rec_media', 'f1_media', 'f1_dp', 'tempo_s']].round(4))

In [ ]:
# No Colab: baixa figuras e resultados num .zip
try:
    from google.colab import files
    import shutil
    shutil.make_archive('saida_grupo6', 'zip', '.', 'figuras')
    shutil.make_archive('resultados_grupo6', 'zip', '.', 'resultados')
    files.download('saida_grupo6.zip'); files.download('resultados_grupo6.zip')
except ImportError:
    print('Fora do Colab: arquivos em ./figuras e ./resultados')